# <center>Procesamiento Digital de Señales de Audio</center>
### <center>Instituto de Ingeniería Eléctrica - UdelaR</center>
## <center> Overlap-Add y Phase Vocoder</center>

### Cómo correr este notebook

Es posible descargarlo y correrlo localmente en su computadora

Tambien pueden correrlo en Google Colab usando el siguiente link.

<table align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/emidan19/audio-dsp/blob/main/practicos/P2c_ola-phase_vocoder.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Correr en Google Colab</a>
  </td>
</table>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import io, signal
from IPython.display import Audio, display

**NOTA:** *Las siguientes dos celdas solo son necesarias para descargar el archivo de ejemplo. Ignórelas si va a trabajar con sus propios archivos de audio.*

In [ ]:
!pip install wget

In [ ]:
import wget
from pathlib import Path

archivos = ['guitar_lick.wav']

# URL y archivo de salida
url = "https://github.com/emidan19/audio-dsp/blob/main/practicos/Archivos_P2/"
for archivo in archivos:
    # Descargar solo si no existe
    if not Path(archivo).exists():
        wget.download(url + archivo + "?raw=true")
        print(f"\nDescarga completada: {archivo}")
    else:
        print(f"El archivo ya existe: {archivo}")

# Overlap-Add y Phase Vocoder

En este ejercicio se implementa la técnica de _phase vocoder_ y se la utiliza para generar transformaciones de la señal de audio. Se sugiere utilizar el archivo _guitar\_lick.wav_ para probarla.

En la etapa de análisis se calcula la transformada de Fourier de tiempo corto (STFT), como

\begin{equation*}
    X_{n_a^u}(e^{j\omega_k})=\sum_{m=-\infty}^{\infty}w_a[m-n_a^u]\, x[m]\, e^{-j\omega_k(m-n_a^u)}
\end{equation*}

en donde, $w_a[n]$ es la ventana de análisis (de largo $L$), $\omega_k=\frac{2\pi}{N} k$, con $N$ la cantidad de puntos de la DFT, y $n_a^u = u \, R_a$, con $R_a$ el hop de análisis en muestras y $u$ el índice de la trama temporal, de valor inicial 0.

En la etapa de síntesis se reconstruye la señal en el dominio del tiempo mediante la antitransformada de Fourier de cada trama temporal y el procedimiento de solapamiento y suma (_overlap-add_), como

\begin{equation*}
    y[n]=\sum_{u=-\infty}^{\infty}w_s[n-n_s^u]\,y_u[n-n_s^u]
\end{equation*}

con

\begin{equation*}
    y_u[n]=\frac{1}{N}\sum_{k=0}^{N-1}Y_{n_s^u}(e^{j\omega_k}) \, e^{j\omega_kn}
\end{equation*}

en donde, $w_s[n]$ es la ventana de síntesis, y $n_s^u=u \, R_s$, siendo $R_s$ el hop de síntesis en muestras. Notar que $y_u[n]$ es la transformada inversa de Fourier de una trama de la STFT. Cuando no hay modificaciones entre la etapa de análisis y síntesis, $Y_{n_s^u}(e^{j\omega_k}) = X_{n_a^u}(e^{j\omega_k})$ y $R_s = R_a$. En ese caso la ventana de síntesis $w_s[n]$ es opcional, pero se hace importante si se aplican modificaciones, por ejemplo cuando $R_s \neq R_a$.

A continuación se carga la señal de audio que será procesada.

In [ ]:
fs, x = io.wavfile.read('guitar_lick.wav')

# normalizar la amplitud máxima (absoluta)
x = x / np.max(np.abs(x)) * 0.9

# tiempo correspondiente a la señal de audio
time_x = np.arange(x.size) / fs

plt.figure(figsize=(12,3))
plt.plot(time_x, x)
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.title('Señal original')
plt.show()

Audio(x, rate=fs)

## Parte 1 - Análisis: STFT con hop $R_a$

Complete la función `analysis_STFT` que implementa la etapa de análisis del _phase vocoder_, es decir una STFT con hop de análisis $R_a$:

1. Obtener cada trama de señal a partir de su muestra inicial $n_a^u = u\,R_a$ y aplicarle la ventana de análisis.
2. Calcular la DFT de cada trama enventanada.
3. Calcular el valor de frecuencia de cada bin en radianes, $\omega_k = \frac{2\pi}{N}k$.
4. Calcular el instante temporal (en muestras) del centro de cada trama.

In [ ]:
def analysis_STFT(x, L=2048, Ra=256, win='hann'):
    '''STFT de la señal de audio de entrada

    Parámetros
    ----------
    x : numpy array
        señal de audio de entrada (mono) como un arreglo numpy 1D.
    L : int
        longitud de la ventana en muestras.
    Ra : int
        hop de análisis en muestras.
    win : string
          tipo de ventana definido en scipy.signal.windows.

    Retorna
    -------
    X_stft : numpy array
             STFT de x como un arreglo numpy 2D (bins x tramas).
    omega_stft : numpy array
                 valores de frecuencia en radianes.
    samps_stft : numpy array
                 muestra temporal correspondiente al centro de cada trama.
    '''

    # longitud de la señal de entrada
    num_samps = x.size

    # ventana de análisis
    window = signal.windows.get_window(win, L)

    # número total de tramas de análisis
    num_frames = int(np.floor((num_samps - L) / Ra))

    # inicializar la STFT
    X_stft = np.zeros((L, num_frames), dtype=complex)

    # procesar cada trama
    for ind in range(num_frames):

        # COMPLETAR: puntos inicial y final de la trama
        n_ini = ...
        n_end = ...

        # COMPLETAR: trama de la señal enventanada
        xr = ...

        # COMPLETAR: guardar la DFT de la trama de la señal
        X_stft[:, ind] = ...

    # Valores de frecuencia en radianes
    omega_stft = 2 * np.pi * np.arange(L) / L

    # COMPLETAR: muestra temporal correspondiente al centro de cada trama
    samps_stft = ...

    return X_stft, omega_stft, samps_stft

Una vez completada la función, calcule la STFT de la señal y visualice el espectrograma. Pruebe distintos valores de $L$ y $R_a$.

In [ ]:
# largo de ventana en muestras
L = 2048
# hop de análisis en muestras
Ra = L // 4

X_stft, omega_stft, samps_stft = analysis_STFT(x, L=L, Ra=Ra, win='hann')

# índice de frecuencia máxima (fs/2)
ind_fmax = X_stft.shape[0] // 2 + 1
# valores de frecuencia (Hz)
stft_freqs = omega_stft[:ind_fmax] * fs / (2 * np.pi)
# valores temporales de la STFT (s)
stft_time = samps_stft / fs

plt.figure(figsize=(12,4))
plt.pcolormesh(stft_time, stft_freqs, 20*np.log10(np.abs(X_stft[:ind_fmax, :]) + 1e-12), cmap='jet', shading='auto')
plt.ylabel('Frecuencia (Hz)')
plt.xlabel('Tiempo (s)')
plt.title(f'Espectrograma (L = {L}, Ra = {Ra})')
plt.show()

**_Responder:_** ¿Qué controla el parámetro $L$? ¿Qué controla el parámetro $R_a$?

_Respuesta:_ 

...


Para usar en el resto del notebook, se define una función auxiliar que calcula y grafica el espectrograma de una señal.

In [ ]:
def plot_spectrogram(y, fs, ax=None, title='', L=2048, Ra=512, fmax=None):
    '''Calcula la STFT de y con analysis_STFT y grafica su espectrograma en dB'''
    X_stft, omega_stft, samps_stft = analysis_STFT(y, L=L, Ra=Ra, win='hann')
    ind_fmax = X_stft.shape[0] // 2 + 1
    stft_freqs = omega_stft[:ind_fmax] * fs / (2 * np.pi)
    stft_time = samps_stft / fs
    X_db = 20 * np.log10(np.abs(X_stft[:ind_fmax, :]) + 1e-12)

    if ax is None:
        _, ax = plt.subplots(figsize=(12,4))
    ax.pcolormesh(stft_time, stft_freqs, X_db, cmap='jet', shading='auto', vmin=X_db.max()-80)
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Frecuencia (Hz)')
    ax.set_title(title)
    if fmax is not None:
        ax.set_ylim([0, fmax])
    return ax

## Parte 2 - Síntesis: overlap-add con hop $R_s$

Complete la función `synthesis_STFT` que implementa la etapa de síntesis a partir de la STFT, usando un hop de síntesis $R_s$ (sin ventana de síntesis):

1. Calcular la reconstrucción de cada trama de señal (antitransformada de la DFT).
2. Acumular tramas sucesivas, ubicadas en $n_s^u = u\,R_s$, según el método de **overlap-add**.
3. Compensar la amplitud de la señal obtenida por el factor de escalado del enventanado $C$.

_Nota:_ si la suma de ventanas desplazadas es constante, $\sum_u w[n - uR] = C$, entonces $C = \frac{1}{R}\sum_{n} w[n]$.

In [ ]:
def synthesis_STFT(X_stft, L=2048, Rs=256, win='hann'):
    '''Síntesis mediante la IFFT de cada trama combinada con overlap-add

    Parámetros
    ----------
    X_stft : numpy array
             STFT como un arreglo numpy 2D (bins x tramas).
    L : int
        longitud de la ventana en muestras.
    Rs : int
        hop de síntesis en muestras.
    win : string
          tipo de ventana definido en scipy.signal.windows.

    Retorna
    -------
    y : numpy array
        señal de audio de salida (mono) como un arreglo numpy 1D.
    '''

    # ventana de análisis
    window = signal.windows.get_window(win, L)

    # número total de tramas
    num_frames = X_stft.shape[1]

    # inicializar la señal de salida en el dominio del tiempo
    y = np.zeros(num_frames * Rs + L)

    # procesar cada trama
    for ind in range(num_frames):

        # COMPLETAR: trama de señal reconstruida
        yr = ...

        # COMPLETAR: puntos inicial y final de la trama
        n_ini = ...
        n_end = ...

        # COMPLETAR: overlap-add de la trama de señal
        y[n_ini:n_end] += ...

    # COMPLETAR: factor de escalado de amplitud
    C = ...

    # compensar el factor de escalado de amplitud
    y /= C

    return y

## Parte 3 - Reconstrucción con $R_a = R_s$

Sin modificaciones de la STFT, la señal se reconstruye perfectamente si la suma de las ventanas de análisis desplazadas es constante (condición de **overlap-add**):

\begin{equation*}
    \sum_{u} w_a[n - uR_a] = C \quad \forall n
\end{equation*}

Para la ventana de Hann (periódica) de largo $L$ esto se cumple para $R_a = L/2, L/4, L/8, \dots$

Realice el análisis y la síntesis con $R_a = R_s$ para dos valores del hop:

* $R_a = R_s = L/4$ (se cumple la condición de overlap-add),
* $R_a = R_s = 3L/4$ (no se cumple la condición de overlap-add).

En cada caso visualice la suma de ventanas desplazadas, compare la forma de onda reconstruida con la original y evalúe la reconstrucción auditivamente.

In [ ]:
def overlap_add_windows(L, R, num_frames, win='hann'):
    '''Suma (overlap-add) de num_frames ventanas desplazadas R muestras'''
    window = signal.windows.get_window(win, L)
    olawin = np.zeros((num_frames - 1) * R + L)
    for ind in range(num_frames):
        olawin[ind*R:ind*R + L] += window
    return olawin


def snr_reconstruccion(x, y, L, R, num_frames):
    '''SNR (dB) de la reconstrucción y respecto a x, en la región donde
    las ventanas se solapan completamente (se descartan los bordes)'''
    n_ini, n_end = L, (num_frames - 1) * R
    e = x[n_ini:n_end] - y[n_ini:n_end]
    return 10 * np.log10(np.sum(x[n_ini:n_end]**2) / np.sum(e**2))

In [ ]:
L = 2048
# hops a probar: el primero cumple overlap-add y el segundo no
R_list = [L // 4, 3 * L // 4]

y_list = []
fig, axs = plt.subplots(len(R_list), 2, figsize=(14, 7))

for (ax_win, ax_sig), R in zip(axs, R_list):
    X_stft, _, _ = analysis_STFT(x, L=L, Ra=R, win='hann')
    y = synthesis_STFT(X_stft, L=L, Rs=R, win='hann')
    y_list.append(y)

    num_frames = X_stft.shape[1]
    window = signal.windows.get_window('hann', L)
    C = np.sum(window) / R
    snr = snr_reconstruccion(x, y, L, R, num_frames)
    print(f'Ra = Rs = {R}: C = {C:.3f}, SNR de reconstrucción = {snr:.1f} dB')

    # suma de las primeras 12 ventanas desplazadas, normalizada por C
    ax_win.plot(overlap_add_windows(L, R, 12) / C)
    ax_win.set_title(f'Overlap-add de ventanas / C  (Ra = {R})')
    ax_win.set_xlabel('Tiempo (muestras)')

    # fragmento de la señal original y la reconstruida
    n = np.arange(int(1.0*fs), int(1.1*fs))
    ax_sig.plot(n / fs, x[n], label='original')
    ax_sig.plot(n / fs, y[n], '--', label='reconstruida')
    ax_sig.set_title(f'Reconstrucción (Ra = Rs = {R})')
    ax_sig.set_xlabel('Tiempo (s)')
    ax_sig.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
for R, y in zip(R_list, y_list):
    print(f'Ra = Rs = {R} muestras')
    display(Audio(y, rate=fs))

**_Responder:_** ¿Por qué con $R_a = R_s = 3L/4$ no se obtiene reconstrucción perfecta? ¿Cómo se manifiesta auditivamente?

_Respuesta:_ 

## Parte 4 - Reconstrucción con $R_a \neq R_s$

Si se sintetiza con un hop $R_s$ distinto al de análisis, la duración de la señal se modifica por un factor $\alpha = R_s / R_a$.

Realice modificaciones de la escala temporal usando $R_s \neq R_a$ con la función `synthesis_STFT`. En particular, pruebe duplicando ($\alpha = 2$) y reduciendo a la mitad ($\alpha = 1/2$) la duración original. Se fija $R_s = L/4$ y se elige $R_a = R_s / \alpha$.

Antes de escuchar el resultado indique qué tipo de modificación temporal espera que se produzca en cada caso. Analice los resultados obtenidos y los artefactos que se introducen.

In [ ]:
L = 2048
Rs = L // 4
alphas = [2, 0.5]   # dilatación y compresión temporal

y_ola = {}
fig, axs = plt.subplots(1, len(alphas) + 1, figsize=(16, 4), sharey=True)
plot_spectrogram(x, fs, ax=axs[0], title='Original', fmax=4000)

for ax, alpha in zip(axs[1:], alphas):
    Ra = int(Rs / alpha)
    X_stft, _, _ = analysis_STFT(x, L=L, Ra=Ra, win='hann')
    y_ola[alpha] = synthesis_STFT(X_stft, L=L, Rs=Rs, win='hann')
    plot_spectrogram(y_ola[alpha], fs, ax=ax, title=f'OLA: Ra = {Ra}, Rs = {Rs} (α = {alpha})', fmax=4000)

plt.tight_layout()
plt.show()

for alpha in alphas:
    print(f'OLA con α = {alpha}: duración {x.size/fs:.2f} s -> {y_ola[alpha].size/fs:.2f} s')
    display(Audio(y_ola[alpha], rate=fs))

Para ver el efecto sobre la forma de onda se compara un fragmento de la señal original con el fragmento correspondiente de la señal dilatada ($\alpha = 2$).

In [ ]:
def plot_fragmento(x, y, alpha, fs, t0=2.0, dur=0.03, title=''):
    '''Compara el fragmento [t0, t0+dur] de x con el fragmento
    correspondiente [alpha*t0, alpha*(t0+dur)] de y'''
    nx = np.arange(int(t0*fs), int((t0 + dur)*fs))
    ny = np.arange(int(alpha*t0*fs), int(alpha*(t0 + dur)*fs))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharey=True)
    ax1.plot(nx / fs, x[nx])
    ax1.set_title('Original')
    ax2.plot(ny / fs, y[ny])
    ax2.set_title(title)
    ax2.set_xlabel('Tiempo (s)')
    plt.tight_layout()
    plt.show()

plot_fragmento(x, y_ola[2], 2, fs, title='OLA sin phase vocoder (α = 2)')

**_Responder:_** ¿Qué artefactos se introducen? ¿A qué se deben?

_Respuesta:_ 

## Parte 5 - Síntesis con phase vocoder

Para evitar la mayoría de los problemas introducidos debido a la inconsistencia de fase, se utiliza el procedimiento de desdoblamiento de fase (_phase unwrapping_).

Asumiendo que existe un solo componente sinusoidal por bin de la DFT, podemos plantear las siguientes ecuaciones para estimar la fase de $Y_{n_s^u}(e^{j\omega_k})$, cuando transformamos la escala temporal utilizando un hop de síntesis $R_s \neq R_a$.

Se calcula el incremento de fase heterodino, a partir del incremento de fase de tramas sucesivas

\begin{equation*}
    \Delta\Phi_k^u=\angle X_{n_a^u}(e^{j\omega_k}) - \angle X_{n_a^{u-1}}(e^{j\omega_k}) - R_a \, \omega_k
\end{equation*}

Notar que el término $R_a\, \omega_k$ es el incremento de fase que cabría esperar si la frecuencia del componente sinusoidal correspondiera exactamente a la frecuencia de análisis.

Se toma el argumento principal de $\Delta\Phi_k^u$ entre $(-\pi, \pi]$, que denominamos $\Delta_p\Phi_k^u$.

Luego se calcula la estimación de la frecuencia instantánea

\begin{equation*}
    \hat{\omega}_k[{n^u_a}] = \omega_k + \frac{1}{R_a} \, \Delta_p\Phi_k^u
\end{equation*}

Finalmente se calcula la fase de $Y_{n_s^u}(e^{j\omega_k})$ utilizando la fórmula de propagación de fase

\begin{equation*}
    \angle Y_{n_s^u}(e^{j\omega_k}) = \angle Y_{n_s^{u-1}}(e^{j\omega_k}) + R_s \, \hat{\omega}_k[{n^u_a}]
\end{equation*}

con $|Y_{n_s^u}(e^{j\omega_k})| = |X_{n_a^u}(e^{j\omega_k})|$ y fase inicial $\angle Y_{n_s^{0}}(e^{j\omega_k}) = \angle X_{n_a^{0}}(e^{j\omega_k})$.

En la síntesis se aplica una ventana de suavizado $w_s[n] = w_a[n]$ (Hann). Tenga en cuenta que esto modifica el factor de escalado: ahora la condición de overlap-add es sobre el producto $w_a[n]\,w_s[n] = w^2[n]$, por lo que $C = \frac{1}{R_s}\sum_n w^2[n]$. Para la ventana de Hann al cuadrado se cumple la condición para $R_s = L/4$ (pero no para $R_s = L/2$). Para profundizar en el estudio de este procedimiento y otras consideraciones sobre la fase se recomienda [2,3].

Complete la función `synthesis_pv_STFT`, que recibe tanto el hop de análisis $R_a$ como el de síntesis $R_s$.

In [ ]:
def synthesis_pv_STFT(X_stft, L=2048, Ra=256, Rs=256, win='hann'):
    '''Síntesis con phase vocoder: propagación de fase + IFFT de cada trama
    combinada con overlap-add usando ventana de síntesis

    Parámetros
    ----------
    X_stft : numpy array
             STFT como un arreglo numpy 2D (bins x tramas).
    L : int
        longitud de la ventana en muestras.
    Ra : int
        hop de análisis en muestras.
    Rs : int
        hop de síntesis en muestras.
    win : string
          tipo de ventana definido en scipy.signal.windows.

    Retorna
    -------
    y : numpy array
        señal de audio de salida (mono) como un arreglo numpy 1D.
    '''

    # cantidad de bins de frecuencia
    N = X_stft.shape[0]

    # ventana de síntesis (igual a la de análisis)
    window = signal.windows.get_window(win, L)

    # número total de tramas
    num_frames = X_stft.shape[1]

    # inicializar la señal de salida en el dominio del tiempo
    y = np.zeros(num_frames * Rs + L)

    # frecuencia de cada bin en radianes
    omega_k = 2 * np.pi * np.arange(N) / N

    # magnitud y fase de la STFT de análisis
    X_mag = np.abs(X_stft)
    X_phase = np.angle(X_stft)

    # fase inicial de la síntesis: la fase de la primera trama de análisis
    Y_phase = X_phase[:, 0].copy()

    # procesar cada trama
    for ind in range(num_frames):

        if ind > 0:
            # COMPLETAR: incremento de fase heterodino
            delta_phi = ...

            # Argumento principal en (-pi, pi]
            delta_phi_p = np.mod(delta_phi + np.pi, 2 * np.pi) - np.pi

            # COMPLETAR: estimación de la frecuencia instantánea
            omega_hat = ...

            # COMPLETAR: propagación de fase
            Y_phase = ...

        # COMPLETAR: STFT modificada (magnitud del análisis, fase propagada)
        Y_stft = ...

        # Trama de señal reconstruida
        yr = np.real(np.fft.ifft(Y_stft,n=L))

        # COMPLETAR: puntos inicial y final de la trama
        n_ini = ...
        n_end = ...

        # COMPLETAR: overlap-add de la trama con ventana de síntesis
        y[n_ini:n_end] += ...

    # COMPLETAR: factor de escalado de amplitud (ventana de análisis y síntesis)
    C = ...

    # compensar el factor de escalado de amplitud
    y /= C

    return y

Como verificación, si $R_a = R_s$ el phase vocoder no debe modificar la señal: la fase propagada coincide (módulo $2\pi$) con la fase de análisis y se obtiene reconstrucción perfecta.

In [ ]:
L = 2048
R = L // 4

X_stft, _, _ = analysis_STFT(x, L=L, Ra=R, win='hann')
y_pv = synthesis_pv_STFT(X_stft, L=L, Ra=R, Rs=R, win='hann')

print(f'Phase vocoder con Ra = Rs = {R}: SNR de reconstrucción = {snr_reconstruccion(x, y_pv, L, R, X_stft.shape[1]):.1f} dB')

Repita las modificaciones de la escala temporal de la Parte 4 ($\alpha = 2$ y $\alpha = 1/2$), ahora con `synthesis_pv_STFT`, y compare los resultados con los obtenidos sin phase vocoder.

In [ ]:
L = 2048
Rs = L // 4
alphas = [2, 0.5]

y_pv = {}
fig, axs = plt.subplots(2, len(alphas), figsize=(14, 8), sharey=True)

for col, alpha in enumerate(alphas):
    Ra = int(Rs / alpha)
    X_stft, _, _ = analysis_STFT(x, L=L, Ra=Ra, win='hann')
    y_pv[alpha] = synthesis_pv_STFT(X_stft, L=L, Ra=Ra, Rs=Rs, win='hann')
    plot_spectrogram(y_ola[alpha], fs, ax=axs[0, col], title=f'OLA sin phase vocoder (α = {alpha})', fmax=4000)
    plot_spectrogram(y_pv[alpha], fs, ax=axs[1, col], title=f'Phase vocoder (α = {alpha})', fmax=4000)

plt.tight_layout()
plt.show()

for alpha in alphas:
    print(f'α = {alpha}: sin phase vocoder / con phase vocoder')
    display(Audio(y_ola[alpha], rate=fs))
    display(Audio(y_pv[alpha], rate=fs))

In [ ]:
plot_fragmento(x, y_ola[2], 2, fs, title='OLA sin phase vocoder (α = 2)')
plot_fragmento(x, y_pv[2], 2, fs, title='Phase vocoder (α = 2)')

## Parte 6 - Aplicaciones del phase vocoder

Usando el phase vocoder implemente las siguientes transformaciones de la señal de audio:

 1. **Transposición en frecuencia (pitch-shifting)**: Se desea una señal de la misma duración que la señal original, pero alterando su contenido espectral. Para ello deben realizarse dos acciones complementarias: un escalamiento en el tiempo y un cambio en la frecuencia de muestreo. Por ejemplo, si se desea subir/bajar el contenido espectral de la señal un semitono, se debe aumentar/disminuir la duración de la señal por un factor de $2^{\frac{1}{12}}$, y luego aumentar/disminuir la frecuencia de muestreo por el mismo factor, de modo de obtener una señal de la misma duración que la original.

 2. **Armonizador**: Utilizando el efecto de transposición en frecuencia sumar a la señal original una versión desplazada una quinta (factor de $2^{\frac{7}{12}}\simeq\frac{3}{2}$).

 3. **Coro (_chorus_)**: Consiste en simular que la señal de voz de un único interprete es entonada por varias voces cantando al unísono (es una variación del efecto de armonización). Se deben superponer varias señales con pequeñas modificaciones de _pitch_ respecto al de la señal original y ligeramente distinto para cada uno. El cambio no debe ser mayor que un cuarto de semitono (factor de $2^{\frac{1}{48}}$).

### Transposición en frecuencia (pitch-shifting)

Se fija $R_s = L/4$ y se elige $R_a = R_s / \text{factor}$, de modo que el phase vocoder escala la duración por $R_s/R_a \approx \text{factor}$. Luego se remuestrea la señal por el factor inverso: interpretada a la frecuencia de muestreo original $f_s$ recupera la duración original y sus frecuencias quedan multiplicadas por $R_s/R_a$.

In [ ]:
def pitch_shifting(x, factor, L=2048):
    '''Cambia el pitch de una señal de audio por el factor dado,
    manteniendo su duración, utilizando phase vocoder

    Entradas:
    x      : señal a modificar (dominio temporal)
    factor : factor de corrimiento de pitch (> 1 sube, < 1 baja)
    L      : largo de ventana (y de la DFT) a usar
    '''
    # COMPLETAR: hops de síntesis y de análisis
    Rs = L // 4
    Ra = ...

    # COMPLETAR: escalamiento temporal por Rs/Ra con phase vocoder
    X_stft, _, _ = analysis_STFT(x, L=L, Ra=Ra, win='hann')
    y = synthesis_pv_STFT(X_stft, L=L, Ra=Ra, Rs=Rs, win='hann')

    # COMPLETAR: remuestrear por Ra/Rs (cambio de frecuencia de muestreo)
    y = ...

    # ajustar al largo de la señal original
    y = np.pad(y, (0, max(0, x.size - y.size)))[:x.size]

    return y

In [ ]:
semitonos = -7
factor = 2**(semitonos / 12)

y_semitone = pitch_shifting(x, factor, L=2048)

fig, axs = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
plot_spectrogram(x, fs, ax=axs[0], title='Señal original', fmax=2000)
plot_spectrogram(y_semitone, fs, ax=axs[1], title=f'Pitch-shifting de {semitonos} semitonos', fmax=2000)
plt.tight_layout()
plt.show()

display(Audio(x, rate=fs))
display(Audio(y_semitone, rate=fs))

### Armonizador

In [ ]:
def harmonizer(x, semitones, mode='up', L=2048):
    '''Suma a la señal original una versión transpuesta semitones
    hacia arriba (mode='up') o hacia abajo (mode='down')'''
    # COMPLETAR: convertir semitonos + modo en un factor de pitch-shift
    semitones = semitones if mode == 'up' else -semitones
    factor = ...

    x_pshift = pitch_shifting(x, factor, L)

    # COMPLETAR: mezclar la señal original con la voz transpuesta
    return x + x_pshift


modo = 'down'
semitono = 7

y_harm = harmonizer(x, semitono, mode=modo)

plot_spectrogram(y_harm, fs, title=f'Armonizador: {semitono} semitonos ({modo})', fmax=2000)
plt.show()

Audio(y_harm, rate=fs)

### Coro (chorus)

In [ ]:
def chorus(x, depth=1, n_voices=6, mix=0.5, L=4096):
    '''Superpone a la señal original n_voices versiones con pequeños
    corrimientos de pitch, entre -depth/4 y +depth/4 semitonos

    Entradas:
    x        : señal a modificar (dominio temporal)
    depth    : profundidad del efecto (1 corresponde a un cuarto de semitono)
    n_voices : cantidad de voces agregadas
    mix      : proporción de la mezcla correspondiente a las voces agregadas
    L        : largo de ventana (se usa un L grande para tener más
               resolución en Ra y así en el factor de pitch)
    '''
    # COMPLETAR: factores de pitch-shift distribuidos en torno a 1
    factores = 2**(np.linspace(-depth, depth, n_voices) / 48)

    y = (1 - mix) * x
    for factor in factores:
        # COMPLETAR: acumular cada voz desplazada, ponderada por mix
        y += mix / n_voices * pitch_shifting(x, factor, L)
    return y


y_chr = chorus(x, depth=1, n_voices=6, mix=0.5)

plot_spectrogram(y_chr, fs, title='Chorus', fmax=2000)
plt.show()

display(Audio(x, rate=fs))
display(Audio(y_chr, rate=fs))

------
# Referencias

[1] L. R. Rabiner and R. W. Schafer, *Theory and Applications of Digital Speech Processing*. Prentice
Hall, 1st ed., 2011.

[2] J. Laroche and M. Dolson, “Improved phase vocoder time-scale modification of audio,” *IEEE Transactions on Speech and Audio processing*, vol. 7, no. 3, pp. 323–332, 1999.

[3] A. Gotzen, N. Bernardin, and D. Arfib., “Traditional implementations of a phase-vocoder: The tricks
of the trade,” in *International Conference on Digital Audio Effects, Italy*, Dec. 2000